<a href="https://colab.research.google.com/github/Slyyzer/Project_Pertama_Phyton_Saya.ipynb/blob/main/My_First_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classic Snake Game
This is a Python implementation of the classic Snake game. In a notebook environment, we can represent the game state and run the game logic loop.

# 🚩 Minesweeper for Colab
This game uses interactive buttons to ensure it works smoothly in the browser without keyboard issues.

In [55]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random

class Minesweeper:
    def __init__(self, size=8, mines=10):
        self.size = size
        self.mines_count = mines
        self.reset_game()

    def reset_game(self):
        self.grid = [[0 for _ in range(self.size)] for _ in range(self.size)]
        self.mines = set()
        self.revealed = set()
        self.flags = set()
        self.game_over = False
        self.won = False
        self.mode = 'DIG' # or 'FLAG'

        # Place mines
        while len(self.mines) < self.mines_count:
            r, c = random.randint(0, self.size-1), random.randint(0, self.size-1)
            self.mines.add((r, c))

        # Calculate numbers
        for r, c in self.mines:
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < self.size and 0 <= nc < self.size and (nr, nc) not in self.mines:
                        self.grid[nr][nc] += 1

    def reveal(self, r, c):
        if (r, c) in self.revealed or (r, c) in self.flags or self.game_over:
            return

        if (r, c) in self.mines:
            self.game_over = True
            return

        self.revealed.add((r, c))
        if self.grid[r][c] == 0:
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < self.size and 0 <= nc < self.size:
                        self.reveal(nr, nc)

        if len(self.revealed) == (self.size * self.size) - len(self.mines):
            self.won = True
            self.game_over = True

    def toggle_flag(self, r, c):
        if (r, c) in self.revealed or self.game_over: return
        if (r, c) in self.flags: self.flags.remove((r, c))
        else: self.flags.add((r, c))

# UI Logic
game = Minesweeper()
buttons = []
output = widgets.Output()
status_label = widgets.Label(value='Game Active')
mode_btn = widgets.ToggleButton(value=False, description='Mode: DIGGING', button_style='primary')

def update_board():
    with output:
        clear_output(wait=True)
        if game.won:
            status_label.value = "🎉 YOU WON!"
        elif game.game_over:
            status_label.value = "💥 BOOM! Game Over."
        else:
            status_label.value = f"Mines: {game.mines_count} | Flags: {len(game.flags)}"

        for r in range(game.size):
            for c in range(game.size):
                btn = buttons[r * game.size + c]
                if (r, c) in game.revealed:
                    btn.description = str(game.grid[r][c]) if game.grid[r][c] > 0 else ''
                    btn.button_style = 'default'
                    btn.disabled = True
                elif (r, c) in game.flags:
                    btn.description = '🚩'
                    btn.button_style = 'warning'
                elif game.game_over and (r, c) in game.mines:
                    btn.description = '💣'
                    btn.button_style = 'danger'
                else:
                    btn.description = ''
                    btn.button_style = 'info'

def on_click(b):
    idx = buttons.index(b)
    r, c = divmod(idx, game.size)
    if mode_btn.value: # Flagging mode
        game.toggle_flag(r, c)
    else:
        game.reveal(r, c)
    update_board()

def on_mode_change(change):
    mode_btn.description = 'Mode: FLAGGING' if change['new'] else 'Mode: DIGGING'
    mode_btn.button_style = 'warning' if change['new'] else 'primary'

mode_btn.observe(on_mode_change, names='value')

# Create Grid
for i in range(game.size * game.size):
    btn = widgets.Button(description='', layout=widgets.Layout(width='40px', height='40px'))
    btn.on_click(on_click)
    buttons.append(btn)

grid_ui = widgets.VBox([
    widgets.HBox([mode_btn, status_label]),
    widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns=f'repeat({game.size}, 40px)'))
])

display(grid_ui, output)
update_board()

Output()

## 🎮 Game Controls & Visualization

Click the cells to reveal them. Use the **Mode** button to switch between digging for mines and flagging potential threats. If you hit a mine, it's game over! Flag all mines or reveal all safe cells to win.

In [57]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def start_new_game():
    # Initialize game logic
    ms_game = Minesweeper(size=10, mines=12)
    btns = []

    # UI Elements
    status = widgets.HTML(value="<div style='padding: 10px; background: #f0f0f0; border-radius: 5px; text-align: center;'><b>Status: Game Active</b></div>")
    mode = widgets.ToggleButton(value=False, description='MODE: DIGGING', button_style='primary', layout=widgets.Layout(width='140px', margin='5px'))
    restart_btn = widgets.Button(description='Restart Game', button_style='success', layout=widgets.Layout(width='140px', margin='5px'))

    def update_ui():
        if ms_game.won:
            status.value = "<div style='padding: 10px; background: #d4edda; border-radius: 5px; text-align: center;'><b style='color:#155724;'>🎉 VICTORY! All safe cells cleared.</b></div>"
        elif ms_game.game_over:
            status.value = "<div style='padding: 10px; background: #f8d7da; border-radius: 5px; text-align: center;'><b style='color:#721c24;'>💥 BOOM! You hit a mine.</b></div>"
        else:
            status.value = f"<div style='padding: 10px; background: #e2e3e5; border-radius: 5px; text-align: center;'><b>Mines: {ms_game.mines_count} | Flags: {len(ms_game.flags)}</b></div>"

        for r in range(ms_game.size):
            for c in range(ms_game.size):
                b = btns[r * ms_game.size + c]
                if (r, c) in ms_game.revealed:
                    val = ms_game.grid[r][c]
                    b.description = str(val) if val > 0 else ''
                    b.button_style = '' # Reset to standard button style
                    b.style.button_color = '#e0e0e0' if (r + c) % 2 == 0 else '#d0d0d0'
                    b.disabled = True
                elif (r, c) in ms_game.flags:
                    b.description = '🚩'
                    b.button_style = 'warning'
                elif ms_game.game_over and (r, c) in ms_game.mines:
                    b.description = '💣'
                    b.button_style = 'danger'
                else:
                    b.description = ''
                    b.button_style = 'info'

    def handle_click(b):
        idx = btns.index(b)
        r, c = divmod(idx, ms_game.size)
        if mode.value: ms_game.toggle_flag(r, c)
        else: ms_game.reveal(r, c)
        update_ui()

    def handle_restart(b):
        clear_output()
        start_new_game()

    mode.observe(lambda c: setattr(mode, 'description', 'MODE: FLAGGING' if c['new'] else 'MODE: DIGGING'), 'value')
    restart_btn.on_click(handle_restart)

    # Create grid with consistent box sizes
    for i in range(ms_game.size * ms_game.size):
        nb = widgets.Button(description='', layout=widgets.Layout(width='32px', height='32px', margin='1px'))
        nb.on_click(handle_click)
        btns.append(nb)

    grid = widgets.GridBox(btns, layout=widgets.Layout(grid_template_columns=f'repeat({ms_game.size}, 34px)', padding='10px', background='#999'))

    ui = widgets.VBox([
        widgets.HBox([mode, restart_btn], layout=widgets.Layout(justify_content='center')),
        status,
        widgets.HBox([grid], layout=widgets.Layout(justify_content='center'))
    ], layout=widgets.Layout(width='400px', border='2px solid #333', padding='10px', background='#f4f4f4'))

    display(ui)
    update_ui()

start_new_game()